# Data Understanding 2: `bureau`

Loans each applicant has held at **other lenders**, as reported to the credit bureau. This is the applicant's external credit history.

Each table is profiled with the same five questions:

1. **Grain:** what does one row represent?
2. **Real rows:** what does the data actually look like?
3. **Types:** what type is each column, and what does it really mean?
4. **Missing data:** where are the blanks, and why?
5. **Ranges:** do the values make sense, and what looks wrong?

## Setup

DuckDB queries the CSV files directly, so no table is loaded fully into memory. Only small, aggregated results are passed to pandas for display. Run the cells top to bottom.

In [1]:
import duckdb
import pandas as pd

BUREAU = 'data/raw/bureau.csv'

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)
APP = 'data/raw/application_train.csv'

---
## Q1. Grain

In [2]:
duckdb.sql(f'''
    SELECT
        COUNT(*)                     AS total_rows,
        COUNT(DISTINCT SK_ID_BUREAU) AS unique_loans,
        COUNT(DISTINCT SK_ID_CURR)   AS unique_customers
    FROM '{BUREAU}'
''').show()

┌────────────┬──────────────┬──────────────────┐
│ total_rows │ unique_loans │ unique_customers │
│   int64    │    int64     │      int64       │
├────────────┼──────────────┼──────────────────┤
│    1716428 │      1716428 │           305811 │
└────────────┴──────────────┴──────────────────┘



How many applicants in `application_train` have no bureau record at all?

In [3]:
duckdb.sql(f'''
    SELECT COUNT(*) AS applicants_without_bureau_record
    FROM '{APP}'
    WHERE SK_ID_CURR NOT IN (SELECT DISTINCT SK_ID_CURR FROM '{BUREAU}')
''').show()

┌──────────────────────────────────┐
│ applicants_without_bureau_record │
│              int64               │
├──────────────────────────────────┤
│                            44020 │
└──────────────────────────────────┘



**Findings**

- 1,716,428 rows = 1,716,428 unique `SK_ID_BUREAU`: **one row = one external loan**.
- 305,811 customers, about **5.6 loans per customer**.
- Joining it straight onto `application_train` would copy each applicant once per loan, inflating counts and default rates. It must be **aggregated to one row per `SK_ID_CURR` first, then LEFT JOINed**.
- **44,020 applicants** have no bureau record. A LEFT JOIN keeps them; an inner join would silently drop them.

---
## Q2. Real rows

The full history of one customer shows how loans relate to each other, which random rows cannot.

In [4]:
bureau_customer = duckdb.sql(f'''
    SELECT *
    FROM '{BUREAU}'
    WHERE SK_ID_CURR = 340229
    ORDER BY DAYS_CREDIT
''').df()
bureau_customer.T

,0,1,2,3,4,5,6,7,8,9,10,11
SK_ID_CURR,340229,340229,340229,340229,340229,340229,340229,340229,340229,340229,340229,340229
SK_ID_BUREAU,5641751,5641752,5641753,5641754,5820942,5820941,5641750,5641755,5820943,5641756,5641757,5641758
CREDIT_ACTIVE,Closed,Closed,Closed,Closed,Sold,Closed,Closed,Closed,Closed,Closed,Active,Active
CREDIT_CURRENCY,currency 1,currency 1,currency 1,currency 1,currency 1,currency 1,currency 1,currency 1,currency 1,currency 1,currency 1,currency 1
DAYS_CREDIT,-2524,-2319,-2071,-1795,-1702,-1393,-1057,-702,-574,-473,-311,-196
CREDIT_DAY_OVERDUE,0,0,0,0,0,0,0,0,0,0,0,0
DAYS_CREDIT_ENDDATE,NaN,-2136.0,-1766.0,-1429.0,-1387.0,-1087.0,-691.0,-337.0,-26.0,-169.0,54.0,-13.0
DAYS_ENDDATE_FACT,-850.0,-1800.0,-1766.0,-1429.0,-77.0,-1089.0,-688.0,-337.0,-26.0,-169.0,NaN,NaN
AMT_CREDIT_MAX_OVERDUE,NaN,3230.235,2184.21,3063.24,58.32,NaN,NaN,0.0,0.0,0.0,0.0,0.0
CNT_CREDIT_PROLONG,0,0,0,0,1,0,0,0,0,0,0,0


**Key columns**

| Column | Meaning |
|---|---|
| `CREDIT_ACTIVE` | Active, Closed, Sold, Bad debt |
| `CREDIT_TYPE` | Consumer credit, credit card, car loan, mortgage... |
| `DAYS_CREDIT` | When the loan was opened |
| `DAYS_CREDIT_ENDDATE` | **Planned** end (can be in the future) |
| `DAYS_ENDDATE_FACT` | **Actual** end |
| `AMT_CREDIT_SUM` / `_DEBT` / `_LIMIT` | Loan size, amount still owed, card limit |
| `AMT_CREDIT_MAX_OVERDUE` | Largest amount ever overdue |
| `CNT_CREDIT_PROLONG` | Number of times the loan was extended |

**Findings: customer 340229**

- 12 loans: 9 closed, 2 active, 1 sold.
- The sold loan was the only one ever extended, ran about **3.6 years past its planned end**, then was sold to collectors.
- Several **closed** loans had overdue amounts of 2,000 to 3,000 at some point. **Closed does not mean paid on time.**
- Comparing planned and actual end dates shows whether a loan went wrong.

---
## Q3. Column types and meaning

In [5]:
duckdb.sql(f'''
    SELECT CREDIT_ACTIVE, COUNT(*) AS loans
    FROM '{BUREAU}'
    GROUP BY CREDIT_ACTIVE
    ORDER BY loans DESC
''').show()

┌───────────────┬─────────┐
│ CREDIT_ACTIVE │  loans  │
│    varchar    │  int64  │
├───────────────┼─────────┤
│ Closed        │ 1079273 │
│ Active        │  630607 │
│ Sold          │    6527 │
│ Bad debt      │      21 │
└───────────────┴─────────┘



In [6]:
duckdb.sql(f'''
    SELECT CREDIT_TYPE, COUNT(*) AS loans
    FROM '{BUREAU}'
    GROUP BY CREDIT_TYPE
    ORDER BY loans DESC
''').show()

┌──────────────────────────────────────────────┬─────────┐
│                 CREDIT_TYPE                  │  loans  │
│                   varchar                    │  int64  │
├──────────────────────────────────────────────┼─────────┤
│ Consumer credit                              │ 1251615 │
│ Credit card                                  │  402195 │
│ Car loan                                     │   27690 │
│ Mortgage                                     │   18391 │
│ Microloan                                    │   12413 │
│ Loan for business development                │    1975 │
│ Another type of loan                         │    1017 │
│ Unknown type of loan                         │     555 │
│ Loan for working capital replenishment       │     469 │
│ Cash loan (non-earmarked)                    │      56 │
│ Real estate loan                             │      27 │
│ Loan for the purchase of equipment           │      19 │
│ Loan for purchase of shares (margin lending) │       4

**Findings**

- 17 columns, 3 of them text, all **categories**: `CREDIT_ACTIVE`, `CREDIT_TYPE`, `CREDIT_CURRENCY`.
- `CREDIT_ACTIVE`: Closed 1,079,273 · Active 630,607 · Sold 6,527 · Bad debt 21.
  - **Active** = current debt load (ability to pay).
  - **Sold / Bad debt** = a lender gave up on repayment (willingness to pay).
- `CREDIT_TYPE`: mostly consumer credit (1,251,615) and credit cards (402,195).
- `CREDIT_CURRENCY` is anonymised and almost always "currency 1": close to useless.
- Bad debt (21 loans) is too rare to use alone. It is combined with Sold, because both **mean the same thing**. Rare categories are merged only when they share a meaning.

---
## Q4. Missing data

In [7]:
bureau_summary = duckdb.sql(f"SUMMARIZE SELECT * FROM '{BUREAU}'").df()
bureau_summary[['column_name', 'column_type', 'null_percentage']].sort_values('null_percentage', ascending=False)

,column_name,column_type,null_percentage
16,AMT_ANNUITY,DOUBLE,71.47
8,AMT_CREDIT_MAX_OVERDUE,DOUBLE,65.51
7,DAYS_ENDDATE_FACT,DOUBLE,36.92
12,AMT_CREDIT_SUM_LIMIT,DOUBLE,34.48
11,AMT_CREDIT_SUM_DEBT,DOUBLE,15.01
6,DAYS_CREDIT_ENDDATE,DOUBLE,6.15
0,SK_ID_CURR,BIGINT,0.00
4,DAYS_CREDIT,BIGINT,0.00
1,SK_ID_BUREAU,BIGINT,0.00
5,CREDIT_DAY_OVERDUE,BIGINT,0.00


**Theory 1:** `DAYS_ENDDATE_FACT` is blank because the loan has not ended yet. If so, blanks should match the Active loans.

In [8]:
duckdb.sql(f'''
    SELECT
        CREDIT_ACTIVE,
        COUNT(*) AS loans,
        SUM(CASE WHEN DAYS_ENDDATE_FACT IS NULL THEN 1 ELSE 0 END) AS actual_end_blank
    FROM '{BUREAU}'
    GROUP BY CREDIT_ACTIVE
    ORDER BY loans DESC
''').show()

┌───────────────┬─────────┬──────────────────┐
│ CREDIT_ACTIVE │  loans  │ actual_end_blank │
│    varchar    │  int64  │      int128      │
├───────────────┼─────────┼──────────────────┤
│ Closed        │ 1079273 │              125 │
│ Active        │  630607 │           628638 │
│ Sold          │    6527 │             4879 │
│ Bad debt      │      21 │               11 │
└───────────────┴─────────┴──────────────────┘



**Theory 2:** `AMT_ANNUITY` is blank for credit cards because they have no fixed monthly payment. If so, cards should be blank far more often than instalment loans.

In [9]:
duckdb.sql(f'''
    SELECT
        CREDIT_TYPE,
        COUNT(*) AS loans,
        ROUND(100.0 * AVG(CASE WHEN AMT_ANNUITY IS NULL THEN 1 ELSE 0 END), 2) AS pct_annuity_blank
    FROM '{BUREAU}'
    GROUP BY CREDIT_TYPE
    ORDER BY loans DESC
''').show()

┌──────────────────────────────────────────────┬─────────┬───────────────────┐
│                 CREDIT_TYPE                  │  loans  │ pct_annuity_blank │
│                   varchar                    │  int64  │      double       │
├──────────────────────────────────────────────┼─────────┼───────────────────┤
│ Consumer credit                              │ 1251615 │             71.45 │
│ Credit card                                  │  402195 │             72.78 │
│ Car loan                                     │   27690 │             67.53 │
│ Mortgage                                     │   18391 │             64.98 │
│ Microloan                                    │   12413 │             49.92 │
│ Loan for business development                │    1975 │             69.62 │
│ Another type of loan                         │    1017 │              82.6 │
│ Unknown type of loan                         │     555 │             82.52 │
│ Loan for working capital replenishment       │    

**Findings**

| Column | Blank | Reading |
|---|---|---|
| `AMT_ANNUITY` | 71.47% | Not reported |
| `AMT_CREDIT_MAX_OVERDUE` | 65.51% | Not reported |
| `DAYS_ENDDATE_FACT` | 36.92% | Doesn't apply (loan still active) |
| `AMT_CREDIT_SUM_LIMIT` | 34.48% | |
| `AMT_CREDIT_SUM_DEBT` | 15.01% | |
| `DAYS_CREDIT_ENDDATE` | 6.15% | |

- `DAYS_ENDDATE_FACT`: 36.92% blank vs 36.74% of loans Active. Confirmed: a loan that has not ended has no actual end date.
- `AMT_CREDIT_MAX_OVERDUE`: some loans show 0, others blank. If blank meant "never late", lenders would write 0, so blank = **not reported** and must not be read as "never late".
- `AMT_ANNUITY`: theory **rejected**. Credit cards are 72.78% blank vs consumer credit 71.45%, and even car loans (67.53%) and mortgages (64.98%) are mostly blank. Blanks reflect reporting gaps, not loan type. Annuity from `application_train` will be used instead.
- **Small groups lie:** types with one or two loans show 0% or 100% blank. Always read a percentage next to its count.

---
## Q5. Ranges and anomalies

In [10]:
bureau_summary[bureau_summary['column_type'] != 'VARCHAR'][['column_name', 'min', 'max', 'avg']]

,column_name,min,max,avg
0,SK_ID_CURR,100001,456255,278214.9336447553
1,SK_ID_BUREAU,5000000,6843457,5924434.489031873
4,DAYS_CREDIT,-2922,0,-1142.107684680045
5,CREDIT_DAY_OVERDUE,0,2792,0.8181665645165425
6,DAYS_CREDIT_ENDDATE,-42060.0,31199.0,510.51736230309615
7,DAYS_ENDDATE_FACT,-42023.0,0.0,-1017.43714806862
8,AMT_CREDIT_MAX_OVERDUE,0.0,115987185.0,3825.4176612731053
9,CNT_CREDIT_PROLONG,0,9,0.00641040579622332
10,AMT_CREDIT_SUM,0.0,585000000.0,354994.59191767144
11,AMT_CREDIT_SUM_DEBT,-4705600.32,170100000.0,137085.1199521607


How many rows break a logical rule?

In [11]:
duckdb.sql(f'''
    SELECT
        COUNT(*) AS total_loans,
        SUM(CASE WHEN DAYS_CREDIT_ENDDATE < DAYS_CREDIT THEN 1 ELSE 0 END)                   AS ends_before_start,
        ROUND(100.0 * AVG(CASE WHEN DAYS_CREDIT_ENDDATE < DAYS_CREDIT THEN 1 ELSE 0 END), 3) AS pct_ends_before_start,
        SUM(CASE WHEN AMT_CREDIT_SUM_DEBT < 0 THEN 1 ELSE 0 END)                             AS negative_debt,
        ROUND(100.0 * AVG(CASE WHEN AMT_CREDIT_SUM_DEBT < 0 THEN 1 ELSE 0 END), 3)           AS pct_negative_debt
    FROM '{BUREAU}'
''').show()

┌─────────────┬───────────────────┬───────────────────────┬───────────────┬───────────────────┐
│ total_loans │ ends_before_start │ pct_ends_before_start │ negative_debt │ pct_negative_debt │
│    int64    │      int128       │        double         │    int128     │      double       │
├─────────────┼───────────────────┼───────────────────────┼───────────────┼───────────────────┤
│     1716428 │               164 │                  0.01 │          8418 │              0.49 │
└─────────────┴───────────────────┴───────────────────────┴───────────────┴───────────────────┘



**Findings**

- `DAYS_CREDIT` reaches back -2,922 days, exactly **8 years**: likely the bureau's history window.
- **Impossible dates (errors):**
  - `DAYS_CREDIT_ENDDATE` from -42,060 (about 115 years ago) to +31,199 (about 85 years ahead).
  - `DAYS_ENDDATE_FACT` min -42,023; `DAYS_CREDIT_UPDATE` min -41,947 and max +372 (updated after the application).
  - Cross-check: loans were opened at most 8 years ago, so they cannot end 115 years ago. A cross-check turns "suspicious" into "impossible".
- **Impossible amounts (errors):** negative debt (min -4,705,600.32) and negative limits (min -586,406). The same value 4,705,600.32 is the minimum debt and the maximum limit, which hints at values entered in the wrong field.
- **Extreme but possible (outliers):** max overdue 116M, annuity 118M, 2,792 days overdue. `CREDIT_DAY_OVERDUE` averages 0.8 days: almost everyone is 0, a few are huge.
- **Size of the problem:** 164 loans end before they start (0.01%); 8,418 have negative debt (0.49%). Small enough to set to NULL. Size decides the fix.

---
## Summary

| Question | `bureau` |
|---|---|
| Grain | One row per external loan (1.72M), ~5.6 per customer; 44,020 applicants have none |
| Real rows | Closed loans can still have overdue history; planned vs actual end shows trouble |
| Types | 3 categorical columns; Sold + Bad debt combined as "bad loans" |
| Missing | Actual end blank = still active; annuity and max overdue not reported |
| Anomalies | Impossible dates and negative amounts, each under 0.5% of rows |

**Carried into feature engineering:** aggregate per `SK_ID_CURR` (loan counts, active loans, bad loans, overdue history), LEFT JOIN to keep no-record applicants, set impossible values to NULL, never read missing overdue as "never late".